# Track A — Bước A2b: Nâng cấp Backbone lên `DINOv3` (`facebook/dinov3-vitb16-pretrain-lvd1689m`) cho `GC10-DET`

- Nhánh: `duytung/06-gc10-backbone-dinov2`
- Yêu cầu: Đã duyệt quyền truy cập `facebook/dinov3-vitb16-pretrain-lvd1689m` và lưu `HF_TOKEN` trong **Colab Secrets** (icon chìa khóa bên trái) hoặc gán tạm ở Cell 4.
- Tự động đo **CPU % peak/avg, RAM peak GB, GPU % peak, GPU VRAM peak MB** và xuất ra `gc10_a2b_dinov3_summary.txt` / `gc10_a2b_dinov3_summary.json` trên Drive.

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu runtime hiện tại đã mount sẵn)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Cập nhật nhánh duytung/06-gc10-backbone-dinov2
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/06-gc10-backbone-dinov2'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu & kiểm tra GPU (bỏ qua nếu runtime đã cài)
!pip uninstall -y -q torchao
!pip install -q -U transformers
!pip install -q numpyro continuum peft datasets scipy scikit-learn psutil
!nvidia-smi

In [ ]:
# 4. Nạp HF_TOKEN (từ Colab Secrets hoặc điền tạm) và chạy nền Bước A2b (DINOv3)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a2b_dinov3.log'
PID_FILE = f'{WORK_DIR}/gc10_a2b_dinov3.pid'

# Ưu tiên đọc từ Colab Secrets (icon chìa khóa bên trái tên 'HF_TOKEN')
hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        hf_token = ''
# Nếu chưa đặt trong Colab Secrets, bạn có thể bỏ dấu # dòng dưới và dán token tạm:
# hf_token = 'hf_...'
assert hf_token.startswith('hf_'), 'Chưa có HF_TOKEN hợp lệ! Hãy thêm vào Colab Secrets hoặc dán tạm vào biến hf_token.'

env = os.environ.copy()
env['HF_TOKEN'] = hf_token
with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a2b_dinov3_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT, env=env)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A2b (DINOv3) với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a2b_dinov3.log'
PID_FILE = f'{WORK_DIR}/gc10_a2b_dinov3.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a2b_dinov3_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A2b (GC10 DINOv3) ---')
    print(open(SUMMARY_OUT).read())